In [ ]:
import pandas as pd
import re
from tqdm import tqdm
import numpy as np

from data_processing.augmentation_similarity_ import DataAugmentation 
from data_processing.augmentation_full import DataAugmentation as DAOnomasticon
from data_processing.augmentation import DataAugmentation as DACombo
from utils import stack_csvs_from_folder

from config import (
    LEXICON_PATH,
    MORPHEMES_PATH,
    ONOMASTICON_PATH,
    INTERNAL_DATASET_INPUTS,
)

np.random.seed(42)


In [40]:
# ==========================================
# 2. INITIALIZE
# ==========================================
print("Initializing DataAugmentor...")
aug = DataAugmentation(
    lexicon_csv_path=str(LEXICON_PATH),
    morphemes_csv_path=str(MORPHEMES_PATH),
    verbose=True
)

aug_ono = DAOnomasticon(
    lexicon_csv_path=str(LEXICON_PATH),
    morphemes_csv_path=str(MORPHEMES_PATH),
    onomasticon_csv_path=str(ONOMASTICON_PATH),
    verbose=True
)


aug_combo = DACombo(
    lexicon_csv_path=str(LEXICON_PATH),
    morphemes_csv_path=str(MORPHEMES_PATH),
    onomasticon_csv_path=str(ONOMASTICON_PATH),
    verbose=True
)


df_test = stack_csvs_from_folder(str(INTERNAL_DATASET_INPUTS))
# DIAGNOSTICS: Check if data loaded correctly
print("-" * 30)
print(f"DEBUG: AUG")
print(f"  - Unique Akkadian PNs in memory: {len(aug.pn_set)}")
print(f"  - Unique Akkadian GNs in memory: {len(aug.gn_set)}")
print(f"  - Prefixes found: {len(aug.affixes)}")
print(f"  - Suffixes found: {len(aug.suffixes)}")

if len(aug.pn_set) == 0:
    print("\nWARNING: No Personal Names (PN) loaded. Check if the 'type' column in your CSV is exactly 'PN'.")
if len(aug.gn_set) == 0:
    print("\nWARNING: No Geo Names (GN) loaded. Check if the 'type' column in your CSV is exactly 'GN'.")

print("-" * 30)
print(f"DEBUG: AUG ONO")
print(f"  - Unique Akkadian PNs in memory: {len(aug_ono.pn_akk_set)}")
print(f"  - Unique Akkadian GNs in memory: {len(aug_ono.gn_set)}")


if len(aug_ono.pn_akk_to_eng) == 0:
    print("\nWARNING: No Personal Names (PN) loaded. Check if the 'type' column in your CSV is exactly 'PN'.")
if len(aug_ono.gn_set) == 0:
    print("\nWARNING: No Geo Names (GN) loaded. Check if the 'type' column in your CSV is exactly 'GN'.")



Initializing DataAugmentor...


------------------------------
DEBUG: AUG
  - Unique Akkadian PNs in memory: 13047
  - Unique Akkadian GNs in memory: 328
  - Prefixes found: 2
  - Suffixes found: 154
------------------------------
DEBUG: AUG ONO
  - Unique Akkadian PNs in memory: 5439
  - Unique Akkadian GNs in memory: 328


In [41]:
sentence_akk = "dumu a-šùr"
sentence_eng = "Aššur's son"

res, akk, eng = aug.swap_names(sentence_akk, sentence_eng, swap_type="pn")
print(res)
print(akk)
print(eng)


res, akk, eng = aug_combo.swap_pn(sentence_akk, sentence_eng)
print(res)
print(akk)
print(eng)

True
dumu ša-lim-ar-dí
Šalim-wardī's son
True
dumu {d}IM-SIPA-ma
Adad-rē'ī's son


In [42]:
print(f"Rows before: {len(df_test)}")
res, akk, eng = aug_ono.name_swap_augmentation(df=df_test, swap_gn=True, swap_pn=True)
print(f"Rows after: {len(res)}")

Rows before: 2090
Starting augmentation on 2090 rows...
Rows after: 2


In [43]:
print(f"Rows before: {len(df_test)}")
res, akk, eng = aug.name_swap_augmentation(df=df_test, swap_gn=True, swap_pn=True)
print(f"Rows after: {len(df_test)}")

Rows before: 2090
Starting augmentation on 2090 rows...
Augmentation complete. Added 1791 new rows.
Rows after: 2090


In [50]:
print("Running ORIGINAL Augmentation (Bank-based)...")
# Note: In your original code, name_swap_augmentation handles both
df_augmented_orig = aug.name_swap_augmentation(df_test.copy(), swap_pn=True, swap_gn=True)
print(f"Original Method: {len(df_test)} -> {len(df_augmented_orig)} rows")

print("-" * 30)

print("Running NEW Onomasticon Augmentation...")
df_augmented_ono = aug_ono.name_swap_augmentation(df_test.copy(), swap_pn=True, swap_gn=True)
print(f"Onomasticon Method: {len(df_test)} -> {len(df_augmented_ono)} rows")

print("-" * 30)

print("Running NEW Onomasticon Augmentation...")
df_augmented_combo = aug_combo.name_swap_augmentation(df_test.copy(), swap_pn=True, swap_gn=True)
print(f"Onomasticon Method: {len(df_test)} -> {len(df_augmented_combo)} rows")


# 4. Compare Results
total_added_orig = len(df_augmented_orig) - len(df_test)
total_added_ono = len(df_augmented_ono) - len(df_test)
total_added_combo = len(df_augmented_combo) - len(df_test)

print("\nSUMMARY:")
print(f"Original Version added: {total_added_orig} rows")
print(f"Onomasticon Version added: {total_added_ono} rows")
print(f"Onomasticon Version added: {total_added_combo} rows")

Running ORIGINAL Augmentation (Bank-based)...
Starting augmentation on 2090 rows...
Augmentation complete. Added 1791 new rows.
Original Method: 2090 -> 3881 rows
------------------------------
Running NEW Onomasticon Augmentation...
Starting augmentation on 2090 rows...
Onomasticon Method: 2090 -> 3554 rows
------------------------------
Running NEW Onomasticon Augmentation...
Starting augmentation on 2090 rows...
Onomasticon Method: 2090 -> 3999 rows

SUMMARY:
Original Version added: 1791 rows
Onomasticon Version added: 1464 rows
Onomasticon Version added: 1909 rows


In [48]:
# IDs to find
target_orig = "2089"
target_aug = "2089-pn_swap"

# Filter the augmented dataframe (which contains both original and new rows)
row_orig = df_augmented_combo[df_augmented_combo['id'].astype(str) == target_orig]
row_aug = df_augmented_combo[df_augmented_combo['id'].astype(str) == target_aug]

row_aug_simple = df_augmented_orig[df_augmented_orig['id'].astype(str) == target_aug]

print(f"{'='*20} ORIGINAL ROW {target_orig} {'='*20}")
if not row_orig.empty:
    print(f"AKKADIAN:    {row_orig.iloc[0]['transliteration']}")
    print(f"ENGLISH:     {row_orig.iloc[0]['translation']}")
else:
    print("Original row not found.")

print(f"\n{'='*20} AUGMENTED ROW {target_aug} {'='*20}")
if not row_aug.empty:
    print(f"AKKADIAN:    {row_aug.iloc[0]['transliteration']}")
    print(f"ENGLISH:     {row_aug.iloc[0]['translation']}")
else:
    print("Augmented row not found. (The augmentation might have failed for this row)")

print(f"\n{'='*20} AUGMENTED ROW {target_aug} {'='*20}")
if not row_aug.empty:
    print(f"AKKADIAN:    {row_aug_simple.iloc[0]['transliteration']}")
    print(f"ENGLISH:     {row_aug_simple.iloc[0]['translation']}")
else:
    print("Augmented row not found. (The augmentation might have failed for this row)")

==================== ORIGINAL ROW 2089 ====================
AKKADIAN:    a-na a-wa-tim a-ni-a-tim kà-ru-um kà-ni-iš i-dì-ni-a-tí-ma IGI GÍR ša a-šùr ší-bu-tí-ni ni-dí-in 
ENGLISH:     The Kanesh colony gave us for these proceedings and we gave our testimony before Aššur's dagger. 

==================== AUGMENTED ROW 2089-pn_swap ====================
AKKADIAN:    a-na a-wa-tim a-ni-a-tim kà-ru-um kà-ni-iš i-dì-ni-a-tí-ma IGI GÍR ša I-ku ší-bu-tí-ni ni-dí-in 
ENGLISH:     The Kanesh colony gave us for these proceedings and we gave our testimony before Iku's dagger. 

==================== AUGMENTED ROW 2089-pn_swap ====================
AKKADIAN:    a-na a-wa-tim a-ni-a-tim kà-ru-um kà-ni-iš i-dì-ni-a-tí-ma IGI GÍR ša en-um-a-šur ší-bu-tí-ni ni-dí-in 
ENGLISH:     The Kanesh colony gave us for these proceedings and we gave our testimony before Ennam-Aššur's dagger. 


In [46]:
df_augmented_combo.tail()

,id,transliteration,translation
3994,2087-pn_swap,a-na a-wa-tim a-ni-a-tim kà-ru-um kà-né-eš i-...,The Kanesh colony gave us for these proceeding...
3995,2087-gn_swap,a-na a-wa-tim a-ni-a-tim kà-ru-um kà-né-eš i-...,The Kanesh colony gave us for these proceeding...
3996,2088-pn_swap,i-na ḫa-ra-nim ni-lá-ak-ma um-ma da-da-a-ma a-...,We will go on a trip and Dadaya said as follow...
3997,2089-pn_swap,a-na a-wa-tim a-ni-a-tim kà-ru-um kà-ni-iš i-d...,The Kanesh colony gave us for these proceeding...
3998,2089-gn_swap,a-na a-wa-tim a-ni-a-tim kà-ru-um kà-ni-iš i-d...,The Kanesh colony gave us for these proceeding...
